# NB_lab_test_trends

Builds `<catalog>.gold.lab_test_trends` (one row per test_name per calendar month) from silver lab_results, enriched with calendar month/quarter names, with a LAG-based month-over-month abnormal-rate trend. Full rebuild per the Silver-to-Gold STTM.

In [0]:
# Parameterisation disabled: catalog is hard-coded as adb_caresync in the SQL cells below
# # Unity Catalog catalog name (STTM placeholder <catalog>)
# dbutils.widgets.text("catalog", "adb_caresync")
# print("catalog =", dbutils.widgets.get("catalog"))

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresync.gold;

CREATE OR REPLACE TABLE adb_caresync.gold.lab_test_trends AS
WITH monthly AS (
  -- Driving table: lab_results grouped by test name and year/month of test_date
  SELECT
    test_name,
    year(test_date)                        AS year,
    month(test_date)                       AS month,
    COUNT(*)                               AS total_tests,
    COUNT_IF(result_status = 'Abnormal')   AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')   AS critical_test_count
  FROM adb_caresync.silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
with_rates AS (
  SELECT
    m.*,
    CAST(ROUND(m.abnormal_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(m.critical_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_trend AS (
  SELECT
    r.*,
    LAG(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
  FROM with_rates r
)
SELECT
  CAST(t.test_name AS STRING)                    AS test_name,
  CAST(t.year AS INT)                            AS year,
  CAST(t.month AS INT)                           AS month,
  CAST(c.month_name AS STRING)                   AS month_name,
  -- calendar has no quarter_name column, so it is derived from calendar.quarter (e.g. 'Q1')
  CAST(concat('Q', c.quarter) AS STRING)         AS quarter_name,
  CAST(t.total_tests AS INT)                     AS total_tests,
  CAST(t.abnormal_test_count AS INT)             AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)             AS critical_test_count,
  t.abnormal_rate_pct,
  t.critical_rate_pct,
  CAST(t.prev_month_abnormal_rate_pct AS DECIMAL(5,2)) AS prev_month_abnormal_rate_pct,
  CASE
    WHEN t.prev_month_abnormal_rate_pct IS NULL THEN NULL
    ELSE CAST(t.abnormal_rate_pct - t.prev_month_abnormal_rate_pct AS DECIMAL(6,2))
  END                                            AS abnormal_rate_change_pct
FROM with_trend t
LEFT JOIN adb_caresync.silver.calendar c
  ON c.date_key = CAST(date_format(make_date(t.year, t.month, 1), 'yyyyMMdd') AS INT);

In [0]:
%sql
-- Grain check: one row per (test_name, year, month); totals reconcile to silver.lab_results
SELECT
  COUNT(*)                                     AS gold_rows,
  COUNT(DISTINCT test_name, year, month)       AS distinct_test_months,
  SUM(total_tests)                             AS sum_total_tests,
  (SELECT COUNT(*) FROM adb_caresync.silver.lab_results) AS source_lab_results,
  COUNT_IF(month_name IS NULL OR quarter_name IS NULL) AS missing_calendar,
  COUNT_IF(prev_month_abnormal_rate_pct IS NULL) AS first_month_rows,
  COUNT(DISTINCT test_name)                    AS distinct_tests
FROM adb_caresync.gold.lab_test_trends;

In [0]:
%sql
SELECT * FROM adb_caresync.gold.lab_test_trends ORDER BY test_name, year, month LIMIT 24;

In [0]:
# DBTITLE 1,Return lab test trends record count
updated_df = spark.table("adb_caresync.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))